# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Srilaya30/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [23]:
from pathlib import Path
import pandas as pd
import numpy as np

repo_root = Path("/content/flyrank-ml-internship")

DATA_PATH = (
    repo_root
    / "data"
    / "raw"
    / "content_refresh_anonymized.csv"
)

OUTPUT_PATH = (
    repo_root
    / "work"
    / "outputs"
    / "baseline_action_score.csv"
)

print("Repository exists:", repo_root.exists())
print("Dataset exists:", DATA_PATH.exists())
print("Dataset path:", DATA_PATH)

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH}"
    )

OUTPUT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

Repository exists: True
Dataset exists: True
Dataset path: /content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv
Dataset loaded successfully!
Shape: (30000, 44)


In [7]:
import pandas as pd
import numpy as np

OUTPUT_PATH = (
    repo_root / "work/outputs/baseline_action_score.csv"
)

OUTPUT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

display(df.head())

Dataset loaded successfully!
Shape: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Baseline rule

I prioritize webpages for content-refresh review using two observable signals:

1. **Staleness:** webpages that have gone longer without an update receive a higher priority.
2. **CTR opportunity:** webpages with search visibility but relatively weak CTR receive a higher priority.

The baseline is a decision-support ranking, not a prediction of future performance.

The score combines the percentile ranks of the two signals:

- 60% weight: staleness
- 40% weight: CTR opportunity

A higher score means the webpage should be reviewed earlier.

### Reason codes

The rule produces one reason code for each webpage:

- `STALE_CONTENT` — staleness is the main reason for the review.
- `CTR_OPPORTUNITY` — weaker CTR is the main reason for the review.

### Action

The action for every ranked webpage is:

`REVIEW_REFRESH`

This is a baseline rule only. It does not use product flags, future-window information, or the trend label as an input.

In [11]:
# ============================================================
# ML-07 - Signal checks
# ============================================================

# Load the dataset
df = pd.read_csv(DATA_PATH)

# Convert signals to numeric
df["days_since_last_update"] = pd.to_numeric(
    df["days_since_last_update"],
    errors="coerce"
)

df["ctr"] = pd.to_numeric(
    df["ctr"],
    errors="coerce"
)

df["impressions_90d"] = pd.to_numeric(
    df["impressions_90d"],
    errors="coerce"
)

# ------------------------------------------------------------
# SIGNAL 1: STALENESS
# ------------------------------------------------------------

df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 30, 90, 180, 365, np.inf],
    labels=[
        "0-30",
        "31-90",
        "91-180",
        "181-365",
        "365+"
    ]
)

staleness_table = (
    df.groupby(
        "staleness_bucket",
        observed=False
    )
    .agg(
        n=("content_id", "size"),
        avg_impressions=("impressions_90d", "mean"),
        avg_ctr=("ctr", "mean")
    )
    .reset_index()
)

print("SIGNAL CHECK 1 — STALENESS")
display(staleness_table)




df["ctr_bucket"] = pd.qcut(
    df["ctr"],
    q=4,
    duplicates="drop"
)

ctr_table = (
    df.groupby(
        "ctr_bucket",
        observed=False
    )
    .agg(
        n=("content_id", "size"),
        avg_impressions=("impressions_90d", "mean"),
        avg_staleness=("days_since_last_update", "mean")
    )
    .reset_index()
)

print("SIGNAL CHECK 2 — CTR")
display(ctr_table)

SIGNAL CHECK 1 — STALENESS


,staleness_bucket,n,avg_impressions,avg_ctr
0,0-30,20480,4199.614062,0.609021
1,31-90,175,6506.748571,0.117543
2,91-180,9171,7486.665140,0.238367
3,181-365,169,1206.893491,3.210828
4,365+,5,8.200000,20.000000


SIGNAL CHECK 2 — CTR


,ctr_bucket,n,avg_impressions,avg_staleness
0,"(-0.001, 0.07]",15224,1910.121322,43.051104
1,"(0.07, 0.29]",7503,9335.023857,51.741970
2,"(0.29, 100.0]",7273,7822.166644,46.654613


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

### Queue construction

For each webpage, I calculate two percentile-based signals:

- `staleness_score`: higher when the webpage has been longer since its last update.
- `ctr_opportunity_score`: higher when CTR is relatively low.

The final baseline score is:

`baseline_score = 0.60 × staleness_score + 0.40 × ctr_opportunity_score`

The queue is ranked from highest to lowest baseline score.

This is intentionally simple and interpretable so that a future ML model can be compared against this baseline.

In [12]:
# ============================================================
# ML-07 - Build ranked queue
# ============================================================

score_df = df.copy()

# Keep rows that can be scored
score_df = score_df.dropna(
    subset=[
        "content_id",
        "client_id",
        "days_since_last_update",
        "ctr"
    ]
).copy()

# ------------------------------------------------------------
# 1. Staleness score
# ------------------------------------------------------------

score_df["staleness_score"] = (
    score_df["days_since_last_update"]
    .rank(
        method="average",
        pct=True
    )
)

# ------------------------------------------------------------
# 2. CTR opportunity score
# Lower CTR = higher opportunity
# ------------------------------------------------------------

score_df["ctr_opportunity_score"] = (
    1
    - score_df["ctr"].rank(
        method="average",
        pct=True
    )
)

# ------------------------------------------------------------
# 3. Baseline score
# ------------------------------------------------------------

score_df["baseline_score"] = (
    0.60 * score_df["staleness_score"]
    + 0.40 * score_df["ctr_opportunity_score"]
)

# ------------------------------------------------------------
# 4. ONE reason code
# ------------------------------------------------------------

score_df["reason_code"] = np.where(
    score_df["staleness_score"]
    >= score_df["ctr_opportunity_score"],
    "STALE_CONTENT",
    "CTR_OPPORTUNITY"
)

# ------------------------------------------------------------
# 5. Action
# ------------------------------------------------------------

score_df["action"] = "REVIEW_REFRESH"

# ------------------------------------------------------------
# 6. Rank
# ------------------------------------------------------------

queue = (
    score_df[
        [
            "content_id",
            "client_id",
            "baseline_score",
            "reason_code",
            "action"
        ]
    ]
    .sort_values(
        ["baseline_score", "content_id"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

queue["rank"] = np.arange(
    1,
    len(queue) + 1
)

queue = queue[
    [
        "rank",
        "content_id",
        "client_id",
        "baseline_score",
        "reason_code",
        "action"
    ]
]

print("Ranked queue shape:", queue.shape)

display(queue.head(20))

Ranked queue shape: (30000, 6)


,rank,content_id,client_id,baseline_score,reason_code,action
0,1,content_55a5b1c46474,client_4ec9599fc2,0.911893,STALE_CONTENT,REVIEW_REFRESH
1,2,content_f6fdf87348f6,client_4ec9599fc2,0.911893,STALE_CONTENT,REVIEW_REFRESH
2,3,content_1b4ec72dafd4,client_4ec9599fc2,0.911843,STALE_CONTENT,REVIEW_REFRESH
3,4,content_8d56efff1e71,client_4ec9599fc2,0.911843,STALE_CONTENT,REVIEW_REFRESH
4,5,content_06e19c6486b0,client_4ec9599fc2,0.911783,STALE_CONTENT,REVIEW_REFRESH
5,6,content_e2b702f4f92b,client_4ec9599fc2,0.911783,STALE_CONTENT,REVIEW_REFRESH
6,7,content_02b0d6e30129,client_19581e27de,0.911723,STALE_CONTENT,REVIEW_REFRESH
7,8,content_6476d1d8c050,client_19581e27de,0.911723,STALE_CONTENT,REVIEW_REFRESH
8,9,content_7a888d3d99c8,client_19581e27de,0.911723,STALE_CONTENT,REVIEW_REFRESH
9,10,content_94991fe6268c,client_19581e27de,0.911723,STALE_CONTENT,REVIEW_REFRESH


In [14]:
OUTPUT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

queue.to_csv(
    OUTPUT_PATH,
    index=False
)

print("CSV successfully written!")
print(OUTPUT_PATH)
print("Rows written:", len(queue))


CSV successfully written!
/content/flyrank-ml-internship/work/outputs/baseline_action_score.csv
Rows written: 30000


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### Top-20 review notes

I reviewed the top 20 webpages as decision-support candidates.

For each page, I considered the action, reason code, strength of the observed signals, and a plausible situation that could make the recommendation wrong.

These notes are manual review observations, not ground-truth labels.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Write the required ML-07 CSV



top20 = (
    queue.head(20)
    .merge(
        score_df[
            [
                "content_id",
                "client_id",
                "days_since_last_update",
                "ctr",
                "impressions_90d",
                "staleness_score",
                "ctr_opportunity_score"
            ]
        ],
        on=[
            "content_id",
            "client_id"
        ],
        how="left"
    )
)

display(top20)

,rank,content_id,client_id,baseline_score,reason_code,action,days_since_last_update,ctr,impressions_90d,staleness_score,ctr_opportunity_score
0,1,content_55a5b1c46474,client_4ec9599fc2,0.911893,STALE_CONTENT,REVIEW_REFRESH,373,0.0,35,0.999967,0.779783
1,2,content_f6fdf87348f6,client_4ec9599fc2,0.911893,STALE_CONTENT,REVIEW_REFRESH,373,0.0,2,0.999967,0.779783
2,3,content_1b4ec72dafd4,client_4ec9599fc2,0.911843,STALE_CONTENT,REVIEW_REFRESH,372,0.0,2,0.999883,0.779783
3,4,content_8d56efff1e71,client_4ec9599fc2,0.911843,STALE_CONTENT,REVIEW_REFRESH,372,0.0,1,0.999883,0.779783
4,5,content_06e19c6486b0,client_4ec9599fc2,0.911783,STALE_CONTENT,REVIEW_REFRESH,334,0.0,10,0.999783,0.779783
5,6,content_e2b702f4f92b,client_4ec9599fc2,0.911783,STALE_CONTENT,REVIEW_REFRESH,334,0.0,30,0.999783,0.779783
6,7,content_02b0d6e30129,client_19581e27de,0.911723,STALE_CONTENT,REVIEW_REFRESH,313,0.0,176,0.999683,0.779783
7,8,content_6476d1d8c050,client_19581e27de,0.911723,STALE_CONTENT,REVIEW_REFRESH,313,0.0,304,0.999683,0.779783
8,9,content_7a888d3d99c8,client_19581e27de,0.911723,STALE_CONTENT,REVIEW_REFRESH,313,0.0,95,0.999683,0.779783
9,10,content_94991fe6268c,client_19581e27de,0.911723,STALE_CONTENT,REVIEW_REFRESH,313,0.0,7,0.999683,0.779783


In [18]:
print("Bottom 20 / weakest-ranked pages:")

weak_picks = queue.tail(20)

display(weak_picks)

Bottom 20 / weakest-ranked pages:


,rank,content_id,client_id,baseline_score,reason_code,action
29980,29981,content_b36ab9f13f2b,client_9f14025af0,0.035387,STALE_CONTENT,REVIEW_REFRESH
29981,29982,content_a1c65f070bad,client_9f14025af0,0.035020,STALE_CONTENT,REVIEW_REFRESH
29982,29983,content_dfce82404813,client_9f14025af0,0.035020,STALE_CONTENT,REVIEW_REFRESH
29983,29984,content_4e95a8389562,client_9f14025af0,0.034440,STALE_CONTENT,REVIEW_REFRESH
29984,29985,content_b96873ca64c1,client_9f14025af0,0.034440,STALE_CONTENT,REVIEW_REFRESH
29985,29986,content_f26233911f33,client_9f14025af0,0.034440,STALE_CONTENT,REVIEW_REFRESH
29986,29987,content_9a7fe374c900,client_9f14025af0,0.034220,STALE_CONTENT,REVIEW_REFRESH
29987,29988,content_006b16e7a2e7,client_9f14025af0,0.034127,STALE_CONTENT,REVIEW_REFRESH
29988,29989,content_4272d3a330a3,client_9f14025af0,0.034127,STALE_CONTENT,REVIEW_REFRESH
29989,29990,content_cfa4d9f1bf0a,client_d4735e3a26,0.034127,STALE_CONTENT,REVIEW_REFRESH


In [17]:
# ============================================================
# ML-07 - Leakage check
# ============================================================

baseline_inputs = [
    "days_since_last_update",
    "ctr"
]

forbidden_inputs = [
    "trend_direction",
    "product_flag",
    "flag",
    "label",
    "target"
]

print("Signals used by baseline:")

for column in baseline_inputs:
    print(" -", column)

leaked_inputs = set(
    baseline_inputs
).intersection(
    forbidden_inputs
)

print("\nForbidden inputs used:", leaked_inputs)

assert len(leaked_inputs) == 0

print("\n✓ No product flags or label-derived inputs used.")
print("✓ No future-window variable used in the scoring formula.")

Signals used by baseline:
 - days_since_last_update
 - ctr

Forbidden inputs used: set()

✓ No product flags or label-derived inputs used.
✓ No future-window variable used in the scoring formula.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak picks

The baseline is intentionally simple and uses only two observable signals. Some pages may therefore be ranked too low or too high.

The main limitation is that staleness and CTR do not capture every reason a webpage may need attention. For example, search intent, seasonality, or recent work not reflected in the available fields could make a ranking look wrong.

### Leakage check

The baseline uses only `days_since_last_update` and `ctr`.

I did not use product flags, `trend_direction`, target labels, or future-window information in the scoring formula.

The score is therefore directional decision-support rather than a prediction of future performance.

In [19]:
# ============================================================
# ML-07 - Final self-check
# ============================================================

print("========== ML-07 SELF-CHECK ==========")

print("Queue rows:", len(queue))

print(
    "Unique content IDs:",
    queue["content_id"].nunique()
)

print(
    "Missing scores:",
    queue["baseline_score"].isna().sum()
)

print(
    "Missing reason codes:",
    queue["reason_code"].isna().sum()
)

print(
    "Missing actions:",
    queue["action"].isna().sum()
)

print(
    "Scores sorted descending:",
    queue["baseline_score"].is_monotonic_decreasing
)

print(
    "CSV exists:",
    OUTPUT_PATH.exists()
)

assert len(queue) > 0
assert queue["baseline_score"].notna().all()
assert queue["reason_code"].notna().all()
assert queue["action"].notna().all()
assert queue["baseline_score"].is_monotonic_decreasing
assert OUTPUT_PATH.exists()

print("\n✓ ML-07 baseline checks passed.")



========== ML-07 SELF-CHECK ==========
Queue rows: 30000
Unique content IDs: 30000
Missing scores: 0
Missing reason codes: 0
Missing actions: 0
Scores sorted descending: True
CSV exists: True

✓ ML-07 baseline checks passed.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.